# Candidate analytical table — SLA risk at purchase time

After the EDA, the next step is to turn the business understanding into a table that can support modeling. The unit of analysis is the order (`order_id`): the goal is to estimate, in the context of the purchase, which orders may have a higher risk of exceeding the promised delivery date.

This is a candidate table, not a final training set. Every variable still needs to pass the temporal-availability contract and the data-quality audit.

## One row per order

An order may combine several items, products, payments, and sellers, but the logistics outcome is recorded by the dataset only at order level: each `order_id` has a single estimated date shown to the customer, a single carrier hand-off date, and a single customer delivery date. No delivery date exists per item or per seller. If the order were split into one row per item or per seller, every row would carry the same delay, the same event would be counted several times, and the outcome would be attributed to each seller without having been observed individually. The relationships are therefore aggregated to one row per `order_id`, which is also the unit of the promise made to the customer.

The aggregation is illustrated by a hypothetical order with three items sold by three different sellers:

| Item | Seller | Seller state | Price | Freight | Weight | Distance to customer |
|---:|:---:|:---:|---:|---:|---:|---:|
| 1 | A | SP | 100 | 15 | 500 g | 400 km |
| 2 | B | MG | 50 | 20 | 2,000 g | 900 km |
| 3 | C | SP | 30 | 10 | 300 g | 420 km |

Composition variables are obtained by counting. The example order receives `item_count = 3`, `distinct_product_count = 3`, `seller_count = 3`, and `origin_state_count = 2`, since two sellers are in SP and one is in MG; because `item_count` counts item rows, two units of the same product are counted as two rows. Monetary and physical values are summed or summarized by the maximum: `total_price` equals 180, `total_freight_value` equals 45, `mean_item_price` equals 60, `total_weight_g` equals 2,800, and `max_weight_g` equals 2,000, while `freight_per_kg` is obtained as total freight divided by total weight in kilograms.

A primary seller is defined as the seller with the highest item value in the order. In the example, seller A is selected and its state is recorded in `seller_state`; ties are broken by identifier order. The Haversine distance (Sinnott, 1984) is computed for each item row and then summarized per order, so `distance_mean_km` is about 573 km. Because the mean is taken over items, a seller with two items receives double weight. The share of items shipped from the customer's own state is recorded in `same_state_share`. Payments, which also form a one-to-many relationship, are summarized by the number of records, the largest installment count, and the most frequent payment type.

Network history variables are first computed for each order–seller pair, using only outcomes known before the purchase, and only then reduced to one row. Breach rates, late-dispatch rate, recency, and *cold start* indicators are summarized by the maximum, because a single seller with an unfavorable history is enough to put the whole order at risk; the risk of seller B is therefore not diluted by the others. History counts are summarized by the minimum, so the order carries the evidence of the least-known seller. A mean weighted by item value is also computed, but it is kept as an auxiliary column outside the model.

The historical network rates (breach rates by seller, by destination, and by seller–destination edge, and late-dispatch rate by seller) are smoothed toward the global rate known before the purchase, with a fixed strength m = 20 (Cestnik, 1990; Micci-Barreca, 2001):

$$
\tilde{r} = \frac{s + m\,p}{n + m}
$$

Where:

- $\tilde{r}$: smoothed rate, used as a feature;
- $s$: number of breaches (or late dispatches) among the combination's historical orders;
- $n$: number of the combination's historical orders with an outcome known before the purchase;
- $p$: as-of global rate (the *prior*), computed on all orders with an outcome known before the purchase;
- $m$: smoothing strength, fixed at 20, which acts as a number of fictitious orders at the global rate.

The formula can be rewritten as a weighted mean of the combination's observed rate, $\hat{r} = s/n$, and the global rate:

$$
\tilde{r} = \frac{n}{n + m}\,\hat{r} + \frac{m}{n + m}\,p
$$

The weight of the observed rate, $n/(n + m)$, grows with the available history. A combination without history ($n = 0$) receives exactly the global rate; with 20 historical orders, the observed and global rates weigh the same; with 180, the observed rate accounts for 90% of the value. A combination with few orders therefore stays close to the global rate, which prevents a single breach from producing a 100% rate, and a combination with many orders converges to its observed rate. The value of $m$ is a documented project choice, not a parameter estimated from the data.

The seller-level reading is not removed by this grain. An auxiliary order–seller table can be used for diagnostics, and the historical late-dispatch rate is estimated only on single-seller orders, the only ones in which the dispatch can be attributed to a specific seller. The reach of the aggregation is limited: among the 96,470 supervised orders, 1,216 have two sellers, 54 have three, and 5 have four or more, according to the count shown in notebook 03. Actual delivery fields are kept outside the candidate table and are used only as the target.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

LANG = "en"
project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / "data").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

from ecom_delivery_sla import (
    build_candidate_order_features,
    build_supervised_orders,
    feature_catalog,
    feature_quality_summary,
    read_olist_tables,
)

tables = read_olist_tables(
    project_root / "data" / "raw" / "olist" / "csv",
    table_names=("orders", "items", "products", "payments", "customers", "sellers", "geolocation"),
)
supervised = build_supervised_orders(tables["orders"])
candidate_features = build_candidate_order_features(
    orders=supervised,
    items=tables["items"],
    products=tables["products"],
    payments=tables["payments"],
    customers=tables["customers"],
    sellers=tables["sellers"],
    geolocation=tables["geolocation"],
)

display(candidate_features.head())
print(f"Orders in candidate table: {len(candidate_features):,}")

,order_id,customer_id,order_purchase_timestamp,order_estimated_delivery_date,customer_state,customer_city,item_count,distinct_product_count,distinct_category_count,total_price,...,edge_hist_median_lead_days,dest_hist_breach_rate,dest_hist_median_lead_days,seller_out_degree,dest_in_degree,promise_slack,seller_hist_breach_rate_value_weighted_mean,edge_hist_breach_rate_value_weighted_mean,seller_hist_dispatch_late_rate_value_weighted_mean,dest_hist_orders
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,2017-10-02 10:56:33,2017-10-18,SP,sao paulo,1,1,1,29.99,...,5.027593,0.028222,7.646088,7.0,990.0,10.516470,0.010751,0.012990,0.028653,9959
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,2018-07-24 20:41:37,2018-08-13,BA,barreiras,1,1,1,118.70,...,7.231088,0.126014,17.452060,10.0,855.0,11.906678,0.034347,0.065234,0.025801,2935
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,2018-08-08 08:38:49,2018-09-04,GO,vianopolis,1,1,1,159.90,...,21.989352,0.064711,14.096389,27.0,672.0,4.650359,0.109852,0.113480,0.050325,1855
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,2017-11-18 19:28:06,2017-12-15,RN,sao goncalo do amarante,1,1,1,45.00,...,NaN,0.073886,17.269387,10.0,122.0,8.919433,0.030904,0.033442,0.048538,165
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,2018-02-13 21:18:39,2018-02-26,SP,santo andre,1,1,1,19.90,...,5.925313,0.033816,7.897743,7.0,1482.0,6.186736,0.054571,0.025676,0.031036,20414


Orders in candidate table: 96,470


### Insight

The table combines order, item, payment, customer, seller, and geolocation attributes into one order-level row. This creates a coherent basis for model comparison, but it does not yet prove that every variable was available at purchase time.

## Availability catalog and leakage risk

The catalog separates candidate variables from variables that require confirmation. `promised_lead_days` uses the estimated date recorded in the order, but its eligibility depends on confirming when that promise was defined. Payments and distance must also be assessed against the actual moment when the operational decision would have been made.

Fields derived from actual delivery or observed delay are not included in this table. They will only be used to evaluate the historical outcome.

The catalog turns a column list into auditable decisions: approved as a candidate, pending temporal confirmation, or reserved for quality control.

In [2]:
catalog = feature_catalog()
display(catalog)
display(catalog.groupby("status", as_index=False).size().rename(columns={"size": "features"}))

,feature,source,status,feature_type,use_in_model,definition
0,order_id,order,quality_control,identifier,False,Order identifier
1,customer_id,customer,quality_control,identifier,False,Order-level customer identifier
2,primary_seller_id,sellers,quality_control,identifier,False,Seller with highest item value in the order
3,order_purchase_timestamp,order,available_at_purchase,datetime,False,Purchase timestamp
4,order_estimated_delivery_date,order,available_at_purchase,datetime,False,Estimated delivery date in the order data
5,promised_lead_days,order,available_at_purchase,numeric,True,Estimated promise minus purchase
6,customer_state,customer,available_at_purchase,categorical,True,Customer destination state
7,customer_is_capital,customer,available_at_purchase,numeric,True,Customer city matches the state capital
8,item_count,order_items,available_at_purchase,numeric,True,Number of item rows
9,distinct_product_count,order_items,available_at_purchase,numeric,True,Number of distinct products


,status,features
0,available_at_approval,2
1,available_at_purchase,48
2,quality_control,9


### Insight

The existence of a column in the source does not prove that it can be used for prediction. Before training, the contract must confirm its availability point, handle missing values, and decide which variables will remain diagnostic only.

## Candidate-table integrity

This check looks at duplicate orders and missing values in aggregated variables. Missingness will not be silently corrected: it will be classified as a real absence of the relationship, lack of geolocation coverage, or a data-quality issue.

The result of this cell defines what still requires a decision before modeling. The table is a technical bridge; the production gate remains open while the temporal contract and anomalies are unresolved.

In [3]:
quality = feature_quality_summary(candidate_features)
display(quality.sort_values(["missing_rate", "feature"], ascending=[False, True]))
print(f"Orders: {len(candidate_features):,}")
print(f"Unique IDs: {candidate_features['order_id'].is_unique}")

,feature,missing,missing_rate,data_type,source,status,feature_type,use_in_model,definition,missing_action
49,edge_hist_median_lead_days,19788,0.205121,float64,network,available_at_purchase,numeric,True,Prior median actual lead time on the seller-to...,no_missing_observed_or_review_if_missing
44,seller_days_since_last,5174,0.053633,float64,network,available_at_purchase,numeric,True,Days since the seller's latest known delivery ...,no_missing_observed_or_review_if_missing
52,seller_out_degree,5174,0.053633,float64,network,available_at_purchase,numeric,True,Number of destination states already served by...,no_missing_observed_or_review_if_missing
28,distance_max_km,476,0.004934,float64,geolocation,quality_control,numeric,False,Maximum distance retained only for redundancy ...,no_missing_observed_or_review_if_missing
27,distance_mean_km,476,0.004934,float64,geolocation,available_at_purchase,numeric,True,Mean approximate seller-customer Haversine dis...,retain_order_impute_on_train_and_add_missing_i...
26,distance_min_km,476,0.004934,float64,geolocation,quality_control,numeric,False,Minimum distance retained only for redundancy ...,no_missing_observed_or_review_if_missing
31,distance_range_km,476,0.004934,float64,geolocation,available_at_purchase,numeric,False,Maximum minus minimum seller-customer distance...,no_missing_observed_or_review_if_missing
51,dest_hist_median_lead_days,285,0.002954,float64,network,available_at_purchase,numeric,True,Prior median actual lead time of the destinati...,no_missing_observed_or_review_if_missing
53,dest_in_degree,285,0.002954,float64,network,available_at_purchase,numeric,True,Number of seller-destination edges already see...,no_missing_observed_or_review_if_missing
54,promise_slack,266,0.002757,float64,network,available_at_purchase,numeric,True,Promised lead time minus prior median lead tim...,no_missing_observed_or_review_if_missing


Orders: 96,470
Unique IDs: True


### Final insight and next step

The candidate table organizes the feature discussion without mixing information observed after the event. The next step is to close the availability contract: decide which columns were known at checkout, formalize missing-value treatment, and separate the supervised population from the future scoring population.

Only after this gate will a temporal split be created and classification models — for `sla_breach` — be compared with regression models — for the number of delay days.

## References

Cestnik, B. (1990). Estimating probabilities: a crucial task in machine learning. *Proceedings of ECAI 1990*, 147–149.

Micci-Barreca, D. (2001). A preprocessing scheme for high-cardinality categorical attributes in classification and prediction problems. *ACM SIGKDD Explorations Newsletter*, 3(1), 27–32. https://doi.org/10.1145/507533.507538

Sinnott, R. W. (1984). Virtues of the Haversine. *Sky and Telescope*, 68(2), 159.